# 04d · Oro: anomalías del día

**OceanWatch Analytics · Entrega 2** · Alimenta la vista "anomalías del día" del tablero
(requisito 4) y el contador de buques con anomalías.

## Entradas y salidas

| | Objeto | Detalle |
|---|---|---|
| **Entrada** | `gold.trayectorias_puntos` | Cada posición con su punto anterior, velocidad implícita y corte de viaje |
| **Entrada** | `silver.calidad_metricas` | Versión de plata por fecha (qué recalcular) |
| **Salida** | `gold.anomalias_dia` | Un evento anómalo por fila |

## Decisiones

| Tipo | Regla | `valor` |
|---|---|---|
| `salto_imposible` | `anomalia_velocidad`: velocidad implícita > 60 nudos entre dos reportes seguidos | Velocidad implícita (kn) |
| `sog_inconsistente` | El buque reporta SOG ≤ 1 nudo pero se movió ≥ 0,5 nm a ≥ 15 nudos implícitos, sin corte de viaje | Velocidad implícita (kn) |
| `mmsi_clonado` | El mismo MMSI aparece en dos `MMSI_Real` en el mismo minuto, a más de 50 nm: no puede ser un solo buque | Número de `MMSI_Real` en conflicto |

- **`sog_inconsistente`** atrapa el caso del MMSI 367638030: SOG medio 0 y velocidad implícita
  media de 18 nudos, con saltos pequeños que el candado cinemático (60 nudos) no detecta. La
  distancia mínima evita marcar el ruido del GPS de un buque amarrado (50 m en 5 s ya son 19 nudos).
- **`mmsi_clonado` exige simultaneidad y distancia**, no solo dos `MMSI_Real`: un buque que cruza
  el borde de una celda H3 res 3 cambia de `MMSI_Real` sin ser un clon.
- **Las posiciones "en tierra" no son anomalía de oro**: con la regla dura de plata nunca llegan
  aquí. El tablero de operación las muestra desde `cuarentena_en_tierra` en las métricas de calidad.
- Mismo patrón que el resto de oro: fechas por versión de plata (`comun.fechas_pendientes`),
  escritura idempotente por fecha (`replaceWhere`) y `_version_silver`.

In [0]:
from pyspark.databricks.sql import functions as dbf
from pyspark.sql import functions as f

import comun
import config

spark = config.obtener_spark()
spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOGO = config.leer_parametro("catalogo", config.CATALOGO_DEFECTO)
RECALCULAR_TODO = config.leer_parametro("recalcular_todo", "false").lower() == "true"
cfg = config.Config(CATALOGO)

T_METRICAS = cfg.tabla("silver", "calidad_metricas")
T_PUNTOS = cfg.tabla("gold", "trayectorias_puntos")
T_ANOMALIAS = cfg.tabla("gold", "anomalias_dia")
print(f"{T_PUNTOS}  ->  {T_ANOMALIAS}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_ANOMALIAS} (
    fecha            DATE      NOT NULL COMMENT 'Fecha UTC del evento',
    tipo             STRING    NOT NULL COMMENT 'salto_imposible | sog_inconsistente | mmsi_clonado',
    MMSI             STRING    NOT NULL COMMENT 'Identificador del buque',
    MMSI_Real        STRING    NOT NULL COMMENT 'MMSI + H3_Macro',
    ts               TIMESTAMP NOT NULL COMMENT 'Momento del evento (mmsi_clonado: primer minuto en conflicto)',
    lat              DOUBLE             COMMENT 'Latitud del evento',
    lon              DOUBLE             COMMENT 'Longitud del evento',
    valor            DOUBLE             COMMENT 'Velocidad implícita (kn) o número de MMSI_Real en conflicto',
    detalle          STRING             COMMENT 'Descripción legible del evento',
    _version_silver  TIMESTAMP NOT NULL COMMENT '_calculado_ts de plata con que se construyó esta fecha',
    CONSTRAINT pk_anomalias_dia PRIMARY KEY (fecha, tipo, MMSI_Real, ts)
)
CLUSTER BY (fecha, tipo)
COMMENT 'Anomalías del día: saltos imposibles, SOG inconsistente con el movimiento y MMSI clonados. Base de la vista de anomalías del tablero.'
""")

FECHAS, VERSIONES = comun.fechas_pendientes(spark, T_METRICAS, [T_ANOMALIAS], RECALCULAR_TODO)
print(f"Fechas por recalcular: {[str(d) for d in FECHAS] or 'ninguna'}")

In [0]:
if FECHAS:
    puntos = spark.table(T_PUNTOS).filter(f.col("fecha").isin(FECHAS))
    evento = lambda tipo: [
        "fecha", f.lit(tipo).alias("tipo"), "MMSI", "MMSI_Real",
        f.col("BaseDateTime").alias("ts"), f.col("LAT").alias("lat"), f.col("LON").alias("lon"),
    ]

    saltos = puntos.filter(f.col("anomalia_velocidad")).select(
        *evento("salto_imposible"),
        f.col("velocidad_implicita_kn").alias("valor"),
        f.format_string("%.1f nm en %.2f h", "distancia_nm", "horas_desde_prev").alias("detalle"),
    )

    sog_inconsistente = puntos.filter(
        (f.col("SOG") <= config.ANOM_SOG_DETENIDO_MAX)
        & (f.col("velocidad_implicita_kn") >= config.ANOM_VEL_IMPLICITA_MIN_KN)
        & (f.col("distancia_nm") >= config.ANOM_DISTANCIA_MIN_NM)
        & ~f.col("corte")
    ).select(
        *evento("sog_inconsistente"),
        f.col("velocidad_implicita_kn").alias("valor"),
        f.format_string("SOG %.1f kn, se movió %.2f nm a %.1f kn", "SOG", "distancia_nm", "velocidad_implicita_kn").alias("detalle"),
    )

    # Clonación: solo MMSI con 2+ MMSI_Real en el día; posición media por minuto y MMSI_Real.
    multiples = (
        puntos.groupBy("fecha", "MMSI").agg(f.count_distinct("MMSI_Real").alias("n"))
        .filter(f.col("n") > 1).select("fecha", "MMSI")
    )
    por_minuto = (
        puntos.join(multiples, ["fecha", "MMSI"])
        .groupBy("fecha", "MMSI", "MMSI_Real", f.date_trunc("minute", "BaseDateTime").alias("minuto"))
        .agg(f.avg("LAT").alias("lat"), f.avg("LON").alias("lon"))
    )
    a, b = por_minuto.alias("a"), por_minuto.alias("b")
    conflictos = (
        a.join(b, (f.col("a.fecha") == f.col("b.fecha")) & (f.col("a.MMSI") == f.col("b.MMSI"))
               & (f.col("a.minuto") == f.col("b.minuto")) & (f.col("a.MMSI_Real") != f.col("b.MMSI_Real")))
        .withColumn("dist_nm", dbf.st_distancesphere(
            dbf.st_point("a.lon", "a.lat"), dbf.st_point("b.lon", "b.lat")) / 1852.0)
        .filter(f.col("dist_nm") > config.ANOM_CLON_DISTANCIA_MIN_NM)
    )
    clonados = (
        conflictos.groupBy(f.col("a.fecha").alias("fecha"), f.col("a.MMSI").alias("MMSI"), f.col("a.MMSI_Real").alias("MMSI_Real"))
        .agg(
            f.min("a.minuto").alias("ts"),
            f.min_by("a.lat", "a.minuto").alias("lat"),
            f.min_by("a.lon", "a.minuto").alias("lon"),
            (f.count_distinct("b.MMSI_Real") + 1).cast("double").alias("valor"),
            f.max("dist_nm").alias("dist_max"),
        )
        .select(
            "fecha", f.lit("mmsi_clonado").alias("tipo"), "MMSI", "MMSI_Real", "ts", "lat", "lon", "valor",
            f.format_string("%d MMSI_Real a la vez, hasta %.0f nm entre sí", f.col("valor").cast("int"), "dist_max").alias("detalle"),
        )
    )

    anomalias = (
        saltos.unionByName(sog_inconsistente).unionByName(clonados)
        .join(comun.df_versiones(spark, FECHAS, VERSIONES), "fecha")
    )
    comun.escribir_fechas(spark, anomalias, T_ANOMALIAS, FECHAS)

In [0]:
print("Eventos y buques por fecha y tipo:")
(
    spark.table(T_ANOMALIAS).groupBy("fecha", "tipo")
    .agg(f.count("*").alias("eventos"), f.count_distinct("MMSI").alias("buques"))
    .orderBy("fecha", "tipo").show(truncate=False)
)
print("Caso de control, MMSI 367638030 (SOG ~0 con velocidad implícita ~18 kn):")
(
    spark.table(T_ANOMALIAS).filter(f.col("MMSI") == "367638030")
    .groupBy("fecha", "tipo").agg(f.count("*").alias("eventos"), f.round(f.avg("valor"), 1).alias("valor_medio"))
    .orderBy("fecha", "tipo").show(truncate=False)
)

In [0]:
base = spark.table(T_PUNTOS).filter(
    (f.col("SOG") <= config.ANOM_SOG_DETENIDO_MAX)
    & (f.col("distancia_nm") >= config.ANOM_DISTANCIA_MIN_NM)
    & ~f.col("corte")
)
umbrales = [10, 15, 20, 30]
base.select(
    *[f.count(f.when(f.col("velocidad_implicita_kn") >= u, 1)).alias(f"eventos_{u}kn") for u in umbrales],
    *[f.count_distinct(f.when(f.col("velocidad_implicita_kn") >= u, f.col("MMSI"))).alias(f"buques_{u}kn") for u in umbrales],
).show(truncate=False)